<a href="https://colab.research.google.com/github/vikrant48/AI-ML-All-AI-related-python-codes-/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install groq sentence-transformers chromadb pymupdf

In [1]:
from google.colab import userdata
groq_api_key = userdata.get('GROQ_API_KEY')

In [ ]:
from groq import Groq
client = Groq(api_key=groq_api_key)
print(client)

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import chromadb

db = chromadb.PersistentClient(path="./resume_db")   # Saves to disk
collection = db.get_or_create_collection(
    name='resumes',
    metadata={'hnsw:space': 'cosine'}   # Use cosine similarity
)

In [ ]:
import fitz  # PyMuPDF

def extract_text_from_pdf(file_path):
    text = ""
    with fitz.open(file_path) as doc:
        for page in doc:
            text += page.get_text()
    return text

In [ ]:
resume_text = extract_text_from_pdf("Resume.pdf")

In [ ]:
import re

def extract_name(text):
    lines = text.split("\n")

    for line in lines[:5]:  # check first 5 lines
        line = line.strip()

        # simple rule: name = line with only alphabets and spaces
        if re.match(r"^[A-Za-z ]{3,}$", line):
            return line

    return "Unknown"

In [ ]:
name = extract_name(resume_text)
print(name)

Vikrant Chauhan


In [ ]:
resumes_list = [
    {
        "id": "1",
        "name": name,
        "years": 3,
        "text": resume_text
    }
]

In [ ]:
for i, resume in enumerate(resumes_list):
    vector = model.encode(resume['text']).tolist()

    collection.add(
        ids=[resume['id']],
        embeddings=[vector],
        documents=[resume['text']],
        metadatas=[{
            'name': resume['name'],
            'years_exp': resume['years']
        }]
    )

print(f"Stored {collection.count()} resumes")

Stored 1 resumes


In [ ]:
data = collection.get()

print(data.keys())

for i in range(len(data['ids'])):
    print(f"\nResume {i+1}")
    print("ID:", data['ids'][i])
    print("Name:", data['metadatas'][i]['name'])
    print("Experience:", data['metadatas'][i]['years_exp'])
    print("Text (first 300 chars):")
    print(data['documents'][i][:300])
    print("-" * 60)

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

Resume 1
ID: 1
Name: Vikrant Chauhan
Experience: 3
Text (first 300 chars):
Vikrant Chauhan
Software Development Engineer | Java | Spring Boot | Microservices
 +91-6386696764
# vikrantchauhan9794@gmail.com
ï vikrant-chauhan-in
§ vikrant48
l Portfolio
Professional Summary
Software Development Engineer specializing in Java, Spring Boot, and React.js to build scalable enterpr
------------------------------------------------------------


In [ ]:
query = """
Looking for a Java backend engineer with 3+ years experience,
Spring Boot, AWS, microservices, REST APIs, and database knowledge
"""

In [ ]:
query_vector = model.encode(query).tolist()
print(query_vector)

[-0.03216887265443802, -0.05090872943401337, -0.004690223839133978, 0.01973717100918293, -0.02738153375685215, -0.05139210820198059, -0.01776975207030773, -0.020468486472964287, -0.08008158206939697, -0.015673015266656876, -0.05277875438332558, -0.046463433653116226, 0.07396957278251648, -0.04375186562538147, 0.02062411420047283, -0.022123122587800026, 0.006418933626264334, -0.025671593844890594, 0.06507657468318939, -0.1690797358751297, -0.03183005750179291, 0.011871818453073502, 0.011188696138560772, -0.0772223249077797, 0.03694194182753563, 0.05188525468111038, 0.0813954547047615, 0.026021409779787064, -0.04353209584951401, -0.06556263566017151, -0.07367647439241409, 0.035704657435417175, 0.016844796016812325, 0.016129225492477417, -0.030495716258883476, 0.1460663080215454, 0.007552893366664648, -0.052478935569524765, 0.019211428239941597, 0.02492528222501278, -0.13246633112430573, -0.012450062669813633, -0.006147564854472876, -0.09625953435897827, -0.0034267569426447153, -0.0695354

In [ ]:
results = collection.query(
    query_embeddings=[query_vector],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)
print(results)

{'ids': [['1']], 'embeddings': None, 'documents': [['Vikrant Chauhan\nSoftware Development Engineer | Java | Spring Boot | Microservices\n\x83 +91-6386696764\n# vikrantchauhan9794@gmail.com\nï vikrant-chauhan-in\n§ vikrant48\nl Portfolio\nProfessional Summary\nSoftware Development Engineer specializing in Java, Spring Boot, and React.js to build scalable enterprise-grade applications.\nExpertise in perfecting backend performance (35% latency reduction) and hardening system security against critical vulnerabilities.\nProficient in delivering RESTful APIs, microservices, and high-concurrency caching strategies. Experienced in Agile and Scrum\nmethodologies with strong object-oriented design and problem-solving background in Data Structures and Algorithms (DS&A).\nTechnical Skills\nLanguages: Java, Python, C++, JavaScript, SQL, HTML5, CSS3\nAI/GenAI: LangChain, LangGraph, RAG Systems, ChromaDB, OpenAI API, Gemini API, Prompt Engineering\nBackend Frameworks: Spring Boot, Spring Security, J

In [ ]:
for i in range(len(results['ids'][0])):
    doc = results['documents'][0][i]
    meta = results['metadatas'][0][i]
    dist = results['distances'][0][i]

    score = 1 - dist  # cosine similarity → higher = better

    print(f"\nRank {i+1}")
    print(f"Score: {score:.3f}")
    print(f"Name: {meta['name']}")
    print(f"Experience: {meta['years_exp']} years")
    print(f"Preview: {doc[:200]}...")
    print("-" * 50)


Rank 1
Score: 0.594
Name: Vikrant Chauhan
Experience: 3 years
Preview: Vikrant Chauhan
Software Development Engineer | Java | Spring Boot | Microservices
 +91-6386696764
# vikrantchauhan9794@gmail.com
ï vikrant-chauhan-in
§ vikrant48
l Portfolio
Professional Summary
Sof...
--------------------------------------------------


In [ ]:
context_parts = []

for i in range(len(results['ids'][0])):
    doc = results['documents'][0][i]
    meta = results['metadatas'][0][i]
    dist = results['distances'][0][i]

    score = 1 - dist

    context_parts.append(
        f"""Candidate {i+1}
Name: {meta['name']}
Experience: {meta['years_exp']} years
Match Score: {score:.3f}

Resume:
{doc[:1000]}
"""
    )

context = "\n\n----------------------\n\n".join(context_parts)

print(context[:1500])

Candidate 1
Name: Vikrant Chauhan
Experience: 3 years
Match Score: 0.594

Resume:
Vikrant Chauhan
Software Development Engineer | Java | Spring Boot | Microservices
 +91-6386696764
# vikrantchauhan9794@gmail.com
ï vikrant-chauhan-in
§ vikrant48
l Portfolio
Professional Summary
Software Development Engineer specializing in Java, Spring Boot, and React.js to build scalable enterprise-grade applications.
Expertise in perfecting backend performance (35% latency reduction) and hardening system security against critical vulnerabilities.
Proficient in delivering RESTful APIs, microservices, and high-concurrency caching strategies. Experienced in Agile and Scrum
methodologies with strong object-oriented design and problem-solving background in Data Structures and Algorithms (DS&A).
Technical Skills
Languages: Java, Python, C++, JavaScript, SQL, HTML5, CSS3
AI/GenAI: LangChain, LangGraph, RAG Systems, ChromaDB, OpenAI API, Gemini API, Prompt Engineering
Backend Frameworks: Spring Boot, Spring 

In [ ]:
groq_response = client.chat.completions.create(
  model="llama-3.1-8b-instant",
  temperature=0.3,
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
            {
            "role": "user",
            "content": f"""
Job Query:
{query}

Candidates:
{context}

Instructions:
1. Rank top candidates
2. Show Name
3. Show Match Score (0-100)
4. Give short reason
"""
        }


  ]
)

print(groq_response.choices[0].message.content)

Based on the provided information, here's the ranking of the candidates:

1. **Vikrant Chauhan**
   - Match Score: 59.4 (0.594)
   - Reason: Vikrant has 3 years of experience, which matches the job requirement. He has expertise in Java, Spring Boot, microservices, and REST APIs, making him a strong candidate for the position. His proficiency in delivering high-concurrency caching strategies and experience with Agile and Scrum methodologies are additional strengths.

Since there's only one candidate provided, Vikrant Chauhan is the top candidate for the Java backend engineer position.
